# Task 2: Initial Technical Indicators

This notebook implements the Task 2 rubric by loading historical stock price data into pandas, checking and handling missing values, and computing technical indicators with TA-Lib.

## Rubric Coverage

- Load historical stock price data into a pandas DataFrame
- Show data quality checks and missing-value handling
- Compute TA-Lib indicators: SMA, EMA, RSI, and MACD
- Visualize indicators overlaid on or alongside closing price data

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import talib

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.technical_indicators import fill_missing_price_data, load_price_data

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Data Loading

The CSV below contains historical OHLCV price data for AAPL. It is loaded into a pandas DataFrame with `Date` parsed as the index.

In [ ]:
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'aapl_historical_prices_sample.csv'
prices = load_price_data(DATA_PATH)

print(f'Rows: {len(prices)}')
print(f'Date range: {prices.index.min().date()} to {prices.index.max().date()}')
prices.head()

In [ ]:
prices.info()

## 2. Data Quality and Missing-Value Handling

The dataset is checked for missing values before indicator calculation. Missing values are documented and filled with forward fill followed by backward fill, which is appropriate for isolated gaps in ordered financial time series.

In [ ]:
missing_before = prices.isna().sum().rename('missing_before')
missing_before[missing_before > 0]

In [ ]:
prices_clean = fill_missing_price_data(prices)
missing_after = prices_clean.isna().sum().rename('missing_after')

pd.concat([missing_before, missing_after], axis=1)

## 3. TA-Lib Technical Indicators

The following cell computes common initial indicators using TA-Lib: simple moving average, exponential moving average, relative strength index, and MACD.

In [ ]:
prices_clean['SMA_10'] = talib.SMA(prices_clean['Close'], timeperiod=10)
prices_clean['EMA_10'] = talib.EMA(prices_clean['Close'], timeperiod=10)
prices_clean['RSI_14'] = talib.RSI(prices_clean['Close'], timeperiod=14)

macd, macd_signal, macd_hist = talib.MACD(
    prices_clean['Close'],
    fastperiod=12,
    slowperiod=26,
    signalperiod=9,
)
prices_clean['MACD'] = macd
prices_clean['MACD_signal'] = macd_signal
prices_clean['MACD_hist'] = macd_hist

prices_clean[['Close', 'SMA_10', 'EMA_10', 'RSI_14', 'MACD', 'MACD_signal']].tail()

## 4. Visualization: SMA and EMA Overlaid on Price

The moving averages smooth daily closing-price noise and help identify short-term trend direction.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(prices_clean.index, prices_clean['Close'], label='Close Price', color='#4C78A8', linewidth=1.8)
ax.plot(prices_clean.index, prices_clean['SMA_10'], label='10-Day SMA (TA-Lib)', color='#F28E2B', linewidth=1.6)
ax.plot(prices_clean.index, prices_clean['EMA_10'], label='10-Day EMA (TA-Lib)', color='#59A14F', linewidth=1.6)
ax.set_title('AAPL Close Price with TA-Lib SMA and EMA')
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.legend()
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 5. Visualization: RSI Alongside Price

RSI is plotted below the price series. The 70 and 30 reference lines mark common overbought and oversold zones.

In [ ]:
fig, (ax_price, ax_rsi) = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

ax_price.plot(prices_clean.index, prices_clean['Close'], label='Close Price', color='#4C78A8')
ax_price.set_title('AAPL Close Price and TA-Lib RSI')
ax_price.set_ylabel('Price (USD)')
ax_price.legend()

ax_rsi.plot(prices_clean.index, prices_clean['RSI_14'], label='14-Day RSI (TA-Lib)', color='#B07AA1')
ax_rsi.axhline(70, color='red', linestyle='--', linewidth=1, label='Overbought 70')
ax_rsi.axhline(30, color='green', linestyle='--', linewidth=1, label='Oversold 30')
ax_rsi.set_xlabel('Date')
ax_rsi.set_ylabel('RSI')
ax_rsi.set_ylim(0, 100)
ax_rsi.legend(loc='lower left')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 6. Visualization: MACD Alongside Price

MACD compares short- and long-term moving averages. Crossovers between the MACD and signal lines are often used as momentum signals.

In [ ]:
fig, (ax_price, ax_macd) = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

ax_price.plot(prices_clean.index, prices_clean['Close'], label='Close Price', color='#4C78A8')
ax_price.set_title('AAPL Close Price and TA-Lib MACD')
ax_price.set_ylabel('Price (USD)')
ax_price.legend()

ax_macd.plot(prices_clean.index, prices_clean['MACD'], label='MACD', color='#E15759')
ax_macd.plot(prices_clean.index, prices_clean['MACD_signal'], label='Signal', color='#76B7B2')
ax_macd.bar(prices_clean.index, prices_clean['MACD_hist'], label='Histogram', color='#BAB0AC', alpha=0.6)
ax_macd.axhline(0, color='black', linewidth=0.8)
ax_macd.set_xlabel('Date')
ax_macd.set_ylabel('MACD')
ax_macd.legend(loc='lower left')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## Initial Findings

The sample period shows AAPL declining from late January into early March, with the 10-day SMA and EMA turning lower after price weakness. RSI moves toward weaker momentum during the selloff, while MACD remains useful for confirming shifts in short-term trend direction.